In [ ]:
import os, sys, csv
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import tensorflow_hub as hub

print("TensorFlow:", tf.__version__)

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


TensorFlow: 2.21.0


In [ ]:
!git clone --depth 1 https://github.com/karolpiczak/ESC-50.git
!git clone https://github.com/asmafatima12/HearMe-AI.git

sys.path.append("HearMe-AI/src")
from audio_processing import preprocess_audio, TARGET_SR

DATA_DIR = "ESC-50"
meta = pd.read_csv(f"{DATA_DIR}/meta/esc50.csv")
target_classes = ["crying_baby", "door_wood_knock", "glass_breaking",
                  "siren", "car_horn", "dog", "clock_alarm"]
meta["label"] = meta["category"].where(meta["category"].isin(target_classes), "background")
print(meta.shape, "| sample rate:", TARGET_SR)

Cloning into 'ESC-50'...
remote: Enumerating objects: 2018, done.
remote: Counting objects: 100% (2018/2018), done.
remote: Compressing objects: 100% (1798/1798), done.
remote: Total 2018 (delta 215), reused 2009 (delta 214), pack-reused 0 (from 0)
Receiving objects: 100% (2018/2018), 615.62 MiB | 29.82 MiB/s, done.
Resolving deltas: 100% (215/215), done.
Updating files: 100% (2011/2011), done.
Cloning into 'HearMe-AI'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 46 (delta 17), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 2.70 MiB | 11.10 MiB/s, done.
Resolving deltas: 100% (17/17), done.
(2000, 8) | sample rate: 16000


In [ ]:
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

class_map_path = yamnet.class_map_path().numpy().decode("utf-8")
with open(class_map_path) as f:
    class_names = [row["display_name"] for row in csv.DictReader(f)]

print("YAMNet knows", len(class_names), "sound classes")

YAMNet knows 521 sound classes


In [ ]:
row = meta[meta["category"] == "siren"].iloc[0]
y_raw, sr_raw = librosa.load(f"{DATA_DIR}/audio/{row['filename']}", sr=None)
wav = preprocess_audio(y_raw, sr_raw)

scores, embeddings, spectrogram = yamnet(wav)

print("scores:", scores.shape)
print("embeddings:", embeddings.shape)

top5 = scores.numpy().mean(axis=0).argsort()[-5:][::-1]
for i in top5:
    print(f"{class_names[i]:35s} {scores.numpy().mean(axis=0)[i]:.3f}")

scores: (10, 521)
embeddings: (10, 1024)
Siren                               0.250
Opera                               0.236
Alarm                               0.226
Classical music                     0.187
Emergency vehicle                   0.182


In [ ]:
for cat in ["crying_baby", "glass_breaking", "car_horn"]:
    r = meta[meta["category"] == cat].iloc[0]
    y_r, sr_r = librosa.load(f"{DATA_DIR}/audio/{r['filename']}", sr=None)
    s, e, _ = yamnet(preprocess_audio(y_r, sr_r))
    m = s.numpy().mean(axis=0)
    print(cat, "->", [class_names[i] for i in m.argsort()[-3:][::-1]])

crying_baby -> ['Speech', 'Animal', 'Chicken, rooster']
glass_breaking -> ['Silence', 'Coin (dropping)', 'Glass']
car_horn -> ['Silence', 'Vehicle horn, car horn, honking', 'Vehicle']


In [ ]:
for i, n in enumerate(class_names):
    if any(k in n.lower() for k in ["alarm", "doorbell", "siren", "smoke", "telephone"]):
        print(i, n)

304 Car alarm
317 Police car (siren)
318 Ambulance (siren)
319 Fire engine, fire truck (siren)
349 Doorbell
382 Alarm
383 Telephone
384 Telephone bell ringing
386 Telephone dialing, DTMF
389 Alarm clock
390 Siren
391 Civil defense siren
393 Smoke detector, smoke alarm
394 Fire alarm


In [ ]:
from tqdm import tqdm

X_mean, X_max, y_labels, folds, filenames = [], [], [], [], []

for _, row in tqdm(meta.iterrows(), total=len(meta)):
    y_raw, sr_raw = librosa.load(f"{DATA_DIR}/audio/{row['filename']}", sr=None)
    wav = preprocess_audio(y_raw, sr_raw)
    _, emb, _ = yamnet(wav)
    emb = emb.numpy()
    X_mean.append(emb.mean(axis=0))
    X_max.append(emb.max(axis=0))
    y_labels.append(row["label"])
    folds.append(row["fold"])
    filenames.append(row["filename"])

X_yam = np.hstack([np.array(X_mean), np.array(X_max)])
y_labels = np.array(y_labels)
folds = np.array(folds)

print("X_yam:", X_yam.shape, "| y:", y_labels.shape, "| folds:", folds.shape)

100%|██████████| 2000/2000 [00:23<00:00, 86.22it/s] 

X_yam: (2000, 2048) | y: (2000,) | folds: (2000,)


In [ ]:
print("NaN:", np.isnan(X_yam).sum(), "| Inf:", np.isinf(X_yam).sum())
print("Classes:", np.unique(y_labels))

NaN: 0 | Inf: 0
Classes: ['background' 'car_horn' 'clock_alarm' 'crying_baby' 'dog'
 'door_wood_knock' 'glass_breaking' 'siren']


In [ ]:
os.makedirs("outputs", exist_ok=True)
np.savez_compressed("outputs/yamnet_features.npz",
                    X=X_yam, y=y_labels, folds=folds,
                    filenames=np.array(filenames))

d = np.load("outputs/yamnet_features.npz")
print({k: d[k].shape for k in d.keys()})

from google.colab import files
files.download("outputs/yamnet_features.npz")

{'X': (2000, 2048), 'y': (2000,), 'folds': (2000,), 'filenames': (2000,)}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np

data = np.load("yamnet_features.npz")
print("Keys in file:", list(data.keys()))

Keys in file: ['X', 'y', 'folds', 'filenames']


In [ ]:
X_emb = data["X"]
y_labels = data["y"]
folds = data["folds"]

print("X_emb:", X_emb.shape)
print("y:", y_labels.shape, "| folds:", folds.shape)
print("Classes:", np.unique(y_labels))

X_emb: (2000, 2048)
y: (2000,) | folds: (2000,)
Classes: ['background' 'car_horn' 'clock_alarm' 'crying_baby' 'dog'
 'door_wood_knock' 'glass_breaking' 'siren']


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             precision_score, classification_report,
                             confusion_matrix)

data = np.load("yamnet_features.npz")
print("Keys:", list(data.keys()))

X = data["X"]
y_labels = data["y"]
folds = data["folds"]
print("X:", X.shape, "| y:", y_labels.shape, "| folds:", folds.shape)

Keys: ['X', 'y', 'folds', 'filenames']
X: (2000, 2048) | y: (2000,) | folds: (2000,)


In [15]:
le = LabelEncoder()
y = le.fit_transform(y_labels)
print(dict(enumerate(le.classes_)))

train_mask = np.isin(folds, [1, 2, 3])
val_mask = folds == 4
test_mask = folds == 5

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)

{0: np.str_('background'), 1: np.str_('car_horn'), 2: np.str_('clock_alarm'), 3: np.str_('crying_baby'), 4: np.str_('dog'), 5: np.str_('door_wood_knock'), 6: np.str_('glass_breaking'), 7: np.str_('siren')}
Train: (1200, 2048) | Val: (400, 2048) | Test: (400, 2048)


In [16]:
scaler = StandardScaler().fit(X_train)       # learns from training data only
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(weights))
print({le.classes_[k]: round(v, 2) for k, v in class_weight.items()})

{np.str_('background'): np.float64(0.15), np.str_('car_horn'): np.float64(6.25), np.str_('clock_alarm'): np.float64(6.25), np.str_('crying_baby'): np.float64(6.25), np.str_('dog'): np.float64(6.25), np.str_('door_wood_knock'): np.float64(6.25), np.str_('glass_breaking'): np.float64(6.25), np.str_('siren'): np.float64(6.25)}


In [20]:
model = keras.Sequential([
    keras.layers.Input(shape=(X.shape[1],)),   # reads 2048 from your data
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(8, activation="softmax"),
])

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 526,600 (2.01 MB)

 Trainable params: 526,600 (2.01 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True)

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=100, batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=0,
)
print("Stopped after", len(history.history["loss"]), "epochs")
print("Best val loss:", round(min(history.history["val_loss"]), 3))

Stopped after 16 epochs
Best val loss: 0.385


In [22]:
pred = model.predict(X_test_s, verbose=0).argmax(axis=1)

print("Accuracy :", round(accuracy_score(y_test, pred), 3))
print("Macro F1 :", round(f1_score(y_test, pred, average="macro", zero_division=0), 3))
print()
print(classification_report(y_test, pred, target_names=le.classes_, zero_division=0))

Accuracy : 0.907
Macro F1 : 0.794

                 precision    recall  f1-score   support

     background       1.00      0.91      0.95       344
       car_horn       0.89      1.00      0.94         8
    clock_alarm       0.62      1.00      0.76         8
    crying_baby       0.80      1.00      0.89         8
            dog       0.23      1.00      0.37         8
door_wood_knock       0.80      1.00      0.89         8
 glass_breaking       1.00      0.38      0.55         8
          siren       1.00      1.00      1.00         8

       accuracy                           0.91       400
      macro avg       0.79      0.91      0.79       400
   weighted avg       0.97      0.91      0.93       400



In [23]:
cm = confusion_matrix(y_test, pred)
print(pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_string())

                 background  car_horn  clock_alarm  crying_baby  dog  door_wood_knock  glass_breaking  siren
background              312         1            5            2   22                2               0      0
car_horn                  0         8            0            0    0                0               0      0
clock_alarm               0         0            8            0    0                0               0      0
crying_baby               0         0            0            8    0                0               0      0
dog                       0         0            0            0    8                0               0      0
door_wood_knock           0         0            0            0    0                8               0      0
glass_breaking            0         0            0            0    5                0               3      0
siren                     0         0            0            0    0                0               0      8


In [24]:
def build_model(n_features):
    return keras.Sequential([
        keras.layers.Input(shape=(n_features,)),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(8, activation="softmax"),
    ])

oof_pred = np.zeros_like(y)
fold_f1 = []

for test_f in [1, 2, 3, 4, 5]:
    val_f = test_f % 5 + 1                      # next fold is validation
    train_f = [f for f in range(1, 6) if f not in (test_f, val_f)]

    tr = np.isin(folds, train_f)
    va = folds == val_f
    te = folds == test_f

    sc = StandardScaler().fit(X[tr])            # training folds only
    cw = dict(enumerate(compute_class_weight(
        "balanced", classes=np.unique(y[tr]), y=y[tr])))

    tf.keras.utils.set_random_seed(42)
    m = build_model(X.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(sc.transform(X[tr]), y[tr],
          validation_data=(sc.transform(X[va]), y[va]),
          epochs=100, batch_size=32, class_weight=cw, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(
              monitor="val_loss", patience=10, restore_best_weights=True)])

    oof_pred[te] = m.predict(sc.transform(X[te]), verbose=0).argmax(axis=1)
    fold_f1.append(f1_score(y[te], oof_pred[te], average="macro", zero_division=0))
    print(f"Test fold {test_f}: macro F1 = {fold_f1[-1]:.3f}")

print(f"\nMean = {np.mean(fold_f1):.3f}, std = {np.std(fold_f1):.3f}")

Test fold 1: macro F1 = 0.838
Test fold 2: macro F1 = 0.814
Test fold 3: macro F1 = 0.815
Test fold 4: macro F1 = 0.671
Test fold 5: macro F1 = 0.857

Mean = 0.799, std = 0.066


In [25]:
rows = []
for i, cls in enumerate(le.classes_):
    rows.append({
        "class": cls,
        "recall": recall_score(y, oof_pred, labels=[i], average=None)[0],
        "precision": precision_score(y, oof_pred, labels=[i], average=None, zero_division=0)[0],
    })
print(pd.DataFrame(rows).round(2).to_string(index=False))

print()
print(pd.DataFrame(confusion_matrix(y, oof_pred),
                   index=le.classes_, columns=le.classes_).to_string())

          class  recall  precision
     background    0.92       0.99
       car_horn    0.90       0.69
    clock_alarm    0.88       0.83
    crying_baby    0.92       0.86
            dog    0.90       0.38
door_wood_knock    0.78       0.67
 glass_breaking    0.85       0.47
          siren    0.95       0.88

                 background  car_horn  clock_alarm  crying_baby  dog  door_wood_knock  glass_breaking  siren
background             1588        13            7            6   51               15              35      5
car_horn                  4        36            0            0    0                0               0      0
clock_alarm               3         0           35            0    1                0               1      0
crying_baby               2         1            0           37    0                0               0      0
dog                       2         1            0            0   36                0               1      0
door_wood_knock           4   

In [26]:
!wget -q https://raw.githubusercontent.com/karolpiczak/ESC-50/master/meta/esc50.csv

meta = pd.read_csv("esc50.csv")
cat = meta.set_index("filename").loc[data["filenames"], "category"].values

bg_id = list(le.classes_).index("background")
fa = (y == bg_id) & (oof_pred != bg_id)       # background clips that triggered an alert

out = pd.DataFrame({"predicted_as": le.classes_[oof_pred[fa]],
                    "true_sound": cat[fa]})
print(out.groupby(["predicted_as", "true_sound"]).size()
         .sort_values(ascending=False).head(15).to_string())

predicted_as     true_sound 
glass_breaking   can_opening    13
dog              sneezing        6
glass_breaking   mouse_click     6
dog              coughing        6
                 can_opening     5
                 mouse_click     5
                 cat             5
crying_baby      laughing        4
dog              breathing       4
car_horn         cow             4
dog              rooster         4
door_wood_knock  footsteps       4
                 fireworks       3
glass_breaking   pig             3
dog              crow            3


In [27]:
oof_proba = np.zeros((len(y), 8))

for test_f in [1, 2, 3, 4, 5]:
    val_f = test_f % 5 + 1
    train_f = [f for f in range(1, 6) if f not in (test_f, val_f)]
    tr, va, te = np.isin(folds, train_f), folds == val_f, folds == test_f

    sc = StandardScaler().fit(X[tr])
    cw = dict(enumerate(compute_class_weight(
        "balanced", classes=np.unique(y[tr]), y=y[tr])))

    tf.keras.utils.set_random_seed(42)
    m = build_model(X.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(sc.transform(X[tr]), y[tr],
          validation_data=(sc.transform(X[va]), y[va]),
          epochs=100, batch_size=32, class_weight=cw, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(
              monitor="val_loss", patience=10, restore_best_weights=True)])
    oof_proba[te] = m.predict(sc.transform(X[te]), verbose=0)

print("Done. Check argmax matches earlier result:",
      (oof_proba.argmax(axis=1) == oof_pred).mean())

Done. Check argmax matches earlier result: 1.0


In [28]:
rows = []
for cls in ["dog", "glass_breaking", "door_wood_knock"]:
    i = list(le.classes_).index(cls)
    for t in [0.5, 0.7, 0.8, 0.9, 0.95]:
        alert = oof_proba[:, i] >= t
        tp = (alert & (y == i)).sum()
        rows.append({"class": cls, "threshold": t,
                     "recall": round(tp / (y == i).sum(), 2),
                     "precision": round(tp / max(alert.sum(), 1), 2),
                     "false_alerts": int((alert & (y != i)).sum())})
print(pd.DataFrame(rows).to_string(index=False))

          class  threshold  recall  precision  false_alerts
            dog       0.50    0.90       0.44            46
            dog       0.70    0.88       0.62            21
            dog       0.80    0.88       0.92             3
            dog       0.90    0.82       1.00             0
            dog       0.95    0.80       1.00             0
 glass_breaking       0.50    0.82       0.49            35
 glass_breaking       0.70    0.82       0.72            13
 glass_breaking       0.80    0.72       0.76             9
 glass_breaking       0.90    0.70       0.90             3
 glass_breaking       0.95    0.60       0.96             1
door_wood_knock       0.50    0.78       0.69            14
door_wood_knock       0.70    0.70       0.82             6
door_wood_knock       0.80    0.68       0.82             6
door_wood_knock       0.90    0.62       0.81             6
door_wood_knock       0.95    0.48       0.79             5


In [29]:
rows = []
for cls in ["car_horn", "clock_alarm", "crying_baby", "siren"]:
    i = list(le.classes_).index(cls)
    for t in [0.5, 0.7, 0.8, 0.9, 0.95]:
        alert = oof_proba[:, i] >= t
        tp = (alert & (y == i)).sum()
        rows.append({"class": cls, "threshold": t,
                     "recall": round(tp / (y == i).sum(), 2),
                     "precision": round(tp / max(alert.sum(), 1), 2),
                     "false_alerts": int((alert & (y != i)).sum())})
print(pd.DataFrame(rows).to_string(index=False))

      class  threshold  recall  precision  false_alerts
   car_horn       0.50    0.90       0.73            13
   car_horn       0.70    0.85       0.79             9
   car_horn       0.80    0.78       0.86             5
   car_horn       0.90    0.60       0.89             3
   car_horn       0.95    0.52       0.88             3
clock_alarm       0.50    0.88       0.90             4
clock_alarm       0.70    0.88       0.90             4
clock_alarm       0.80    0.85       0.92             3
clock_alarm       0.90    0.85       0.97             1
clock_alarm       0.95    0.80       1.00             0
crying_baby       0.50    0.92       0.86             6
crying_baby       0.70    0.92       0.88             5
crying_baby       0.80    0.92       0.88             5
crying_baby       0.90    0.92       0.88             5
crying_baby       0.95    0.92       0.88             5
      siren       0.50    0.92       0.90             4
      siren       0.70    0.90       0.90       

In [30]:
epochs_used = []
for test_f in [1, 2, 3, 4, 5]:
    val_f = test_f % 5 + 1
    train_f = [f for f in range(1, 6) if f not in (test_f, val_f)]
    tr, va = np.isin(folds, train_f), folds == val_f
    sc = StandardScaler().fit(X[tr])
    cw = dict(enumerate(compute_class_weight(
        "balanced", classes=np.unique(y[tr]), y=y[tr])))
    tf.keras.utils.set_random_seed(42)
    m = build_model(X.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = m.fit(sc.transform(X[tr]), y[tr],
              validation_data=(sc.transform(X[va]), y[va]),
              epochs=100, batch_size=32, class_weight=cw, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(
                  monitor="val_loss", patience=10, restore_best_weights=True)])
    best = int(np.argmin(h.history["val_loss"])) + 1
    epochs_used.append(best)
    print(f"Fold {test_f}: best epoch = {best}")

print("Median best epoch:", int(np.median(epochs_used)))

Fold 1: best epoch = 28
Fold 2: best epoch = 17
Fold 3: best epoch = 27
Fold 4: best epoch = 6
Fold 5: best epoch = 10
Median best epoch: 17


In [31]:
import joblib, json, os
os.makedirs("models", exist_ok=True)

final_scaler = StandardScaler().fit(X)           # all 2,000 clips
cw_all = dict(enumerate(compute_class_weight(
    "balanced", classes=np.unique(y), y=y)))

tf.keras.utils.set_random_seed(42)
final_model = build_model(X.shape[1])
final_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
final_model.fit(final_scaler.transform(X), y, epochs=17, batch_size=32,
                class_weight=cw_all, verbose=0)
print("Final model trained for 17 epochs.")

Final model trained for 17 epochs.


In [32]:
final_model.save("models/sound_classifier.keras")
joblib.dump(final_scaler, "models/yamnet_scaler.pkl")
joblib.dump(le, "models/label_encoder.pkl")

thresholds = {"background": 1.0, "car_horn": 0.5, "clock_alarm": 0.5,
              "crying_baby": 0.5, "dog": 0.8, "door_wood_knock": 0.5,
              "glass_breaking": 0.7, "siren": 0.5}
with open("models/thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)

print(os.listdir("models"))

['label_encoder.pkl', 'thresholds.json', 'yamnet_scaler.pkl', 'sound_classifier.keras']


In [33]:
m2 = keras.models.load_model("models/sound_classifier.keras")
sc2 = joblib.load("models/yamnet_scaler.pkl")
le2 = joblib.load("models/label_encoder.pkl")

p1 = final_model.predict(final_scaler.transform(X[:50]), verbose=0)
p2 = m2.predict(sc2.transform(X[:50]), verbose=0)
print("Max difference after reload:", np.abs(p1 - p2).max())
print("Label order:", list(le2.classes_))

Max difference after reload: 0.0
Label order: [np.str_('background'), np.str_('car_horn'), np.str_('clock_alarm'), np.str_('crying_baby'), np.str_('dog'), np.str_('door_wood_knock'), np.str_('glass_breaking'), np.str_('siren')]


In [34]:
fname = meta["filename"].iloc[0]
y_raw, sr_raw = librosa.load(f"{DATA_DIR}/audio/{fname}", sr=None)
_, emb, _ = yamnet(preprocess_audio(y_raw, sr_raw))
emb = emb.numpy()

mean_std = np.concatenate([emb.mean(axis=0), emb.std(axis=0)])
saved = np.load("yamnet_features.npz")["X"][0]
print("Max difference (mean+std):", np.abs(mean_std - saved).max())

Max difference (mean+std): 4.279517


In [35]:
saved_all = np.load("yamnet_features.npz")
saved = saved_all["X"][0]
print("Saved row 0 filename:", saved_all["filenames"][0])
print("meta row 0 filename :", meta["filename"].iloc[0])

fname = saved_all["filenames"][0]
y_raw, sr_raw = librosa.load(f"{DATA_DIR}/audio/{fname}", sr=None)

def embed(wav):
    _, e, _ = yamnet(wav)
    return e.numpy()

def report(name, vec):
    d1 = np.abs(vec[:1024] - saved[:1024]).max()
    d2 = np.abs(vec[1024:] - saved[1024:]).max()
    print(f"{name:35s} first half diff = {d1:.5f} | second half diff = {d2:.5f}")

# Recipe 1: our preprocess_audio (normalized), mean then std
e = embed(preprocess_audio(y_raw, sr_raw))
report("normalized, mean+std", np.concatenate([e.mean(0), e.std(0)]))
report("normalized, std ddof=1", np.concatenate([e.mean(0), e.std(0, ddof=1)]))
report("normalized, std then mean", np.concatenate([e.std(0), e.mean(0)]))

# Recipe 2: no peak normalization
y16 = librosa.resample(y_raw, orig_sr=sr_raw, target_sr=16000)
e2 = embed(y16.astype(np.float32))
report("NOT normalized, mean+std", np.concatenate([e2.mean(0), e2.std(0)]))

# Recipe 3: librosa.load directly at 16 kHz
y_d, _ = librosa.load(f"{DATA_DIR}/audio/{fname}", sr=16000)
e3 = embed(y_d.astype(np.float32))
report("librosa sr=16000, mean+std", np.concatenate([e3.mean(0), e3.std(0)]))

Saved row 0 filename: 1-100032-A-0.wav
meta row 0 filename : 1-100032-A-0.wav
normalized, mean+std                first half diff = 0.00000 | second half diff = 4.27952
normalized, std ddof=1              first half diff = 0.00000 | second half diff = 4.18078
normalized, std then mean           first half diff = 1.71622 | second half diff = 5.20831
NOT normalized, mean+std            first half diff = 0.00620 | second half diff = 4.27386
librosa sr=16000, mean+std          first half diff = 0.00620 | second half diff = 4.27386


In [36]:
e = embed(preprocess_audio(y_raw, sr_raw))
second = saved[1024:]

candidates = {
    "max": e.max(0),
    "min": e.min(0),
    "median": np.median(e, axis=0),
    "std": e.std(0),
    "mean of squares": (e ** 2).mean(0),
    "first window": e[0],
    "last window": e[-1],
}
for name, v in candidates.items():
    print(f"{name:18s} diff = {np.abs(v - second).max():.5f}")

max                diff = 0.00000
min                diff = 7.09935
median             diff = 6.10474
std                diff = 4.27952
mean of squares    diff = 28.18122
first window       diff = 6.10474
last window        diff = 6.10474


In [37]:
print("saved second half  first 5:", np.round(second[:5], 3))
print("embedding max       first 5:", np.round(e.max(0)[:5], 3))
print("embedding mean      first 5:", np.round(e.mean(0)[:5], 3))

saved second half  first 5: [2.076 0.393 3.15  0.09  3.411]
embedding max       first 5: [2.076 0.393 3.15  0.09  3.411]
embedding mean      first 5: [1.453 0.275 0.966 0.062 2.388]


In [38]:
saved_all = np.load("yamnet_features.npz")
idxs = [0, 500, 1000, 1500, 1999]
diffs = []
for i in idxs:
    fname = saved_all["filenames"][i]
    y_raw, sr_raw = librosa.load(f"{DATA_DIR}/audio/{fname}", sr=None)
    _, emb, _ = yamnet(preprocess_audio(y_raw, sr_raw))
    emb = emb.numpy()
    vec = np.concatenate([emb.mean(0), emb.max(0)])
    diffs.append(np.abs(vec - saved_all["X"][i]).max())
    print(fname, "max diff =", diffs[-1])

1-100032-A-0.wav max diff = 0.0
2-118624-A-30.wav max diff = 0.0
3-151212-A-24.wav max diff = 0.0
4-196671-A-8.wav max diff = 0.0
5-9032-A-0.wav max diff = 0.0


In [39]:
%%writefile yamnet_features.py
"""YAMNet embedding features for HearMe AI.
Recipe (verified against training data):
    2048 features = [mean over windows (1024), max over windows (1024)]
"""
import numpy as np
import tensorflow_hub as hub
from audio_processing import preprocess_audio

_yamnet = None

def load_yamnet():
    global _yamnet
    if _yamnet is None:
        _yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
    return _yamnet

def yamnet_features(y, sr):
    """Raw audio (any rate) -> 2048-number feature vector."""
    wav = preprocess_audio(y, sr)
    _, emb, _ = load_yamnet()(wav)
    emb = emb.numpy()
    return np.concatenate([emb.mean(axis=0), emb.max(axis=0)])

Writing yamnet_features.py


In [40]:
import importlib, yamnet_features
importlib.reload(yamnet_features)

fname = saved_all["filenames"][0]
y_raw, sr_raw = librosa.load(f"{DATA_DIR}/audio/{fname}", sr=None)
v = yamnet_features.yamnet_features(y_raw, sr_raw)
print("Shape:", v.shape)
print("Max diff vs saved:", np.abs(v - saved_all["X"][0]).max())

Shape: (2048,)
Max diff vs saved: 0.0


In [45]:
from google.colab import files
for f in os.listdir("models"):
    files.download(f"models/{f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Phase 4 Summary: YAMNet Transfer Learning

- Features: YAMNet embeddings (1024 dims per ~1 s window, 10 windows per
  clip), summarized as [mean, max] over windows = 2048 features.
  Verified against the saved file (earlier "mean+std" guess was wrong).
- Classifier: Dense(256, relu) -> Dropout(0.3) -> Dense(8, softmax),
  StandardScaler fit on training folds only, balanced class weights.
- Evaluation: 5-fold CV; each round trains on 3 folds, validates on the
  next fold (early stopping), tests on the held-out fold.
- Macro F1: 0.799 +/- 0.066 (SVM: 0.555 +/- 0.093). YAMNet won 4 of 5 folds.
- Targets caught: 247/280 (88%); background false alerts: 132/1720 (7.7%).
- Weak spots: dog precision 0.38, glass precision 0.47. False alerts come
  mainly from can_opening, mouse_click, sneezing, coughing.
- Per-class thresholds: dog 0.8, glass 0.7, others 0.5 (chosen on CV
  predictions, so optimistic).
- Final model trained on all 2000 clips for 17 epochs (median best epoch;
  varied 6-28). Reported performance is the CV numbers only.
- Saved: sound_classifier.keras, yamnet_scaler.pkl, label_encoder.pkl,
  thresholds.json
- Limitations: no doorbell/fire alarm/phone class; ESC-50 background is
  clean; per-clip rates, not live-stream rates.